# Dataset Check

This notebook summarizes the binary promoter and sigma-factor datasets used by BayesSigma.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_dataset_check import (
    build_class_distribution,
    build_dataset_summary,
    ensure_output_dirs,
    plot_class_distribution,
)
from src.data_loader import check_dataset, load_binary_dataset, load_sigma_dataset, print_dataset_report

## 2. Load Binary Dataset

In [ ]:
binary_train, binary_test = load_binary_dataset(PROJECT_ROOT / "data")
binary_train.head()

## 3. Load Sigma Dataset

In [ ]:
sigma_train, sigma_test = load_sigma_dataset(PROJECT_ROOT / "data")
sigma_train.head()

## 4. Print Binary Train/Test Statistics

In [ ]:
print_dataset_report(binary_train, "Binary Train")
print_dataset_report(binary_test, "Binary Test")

## 5. Print Sigma Train/Test Statistics

In [ ]:
print_dataset_report(sigma_train, "Sigma Train")
print_dataset_report(sigma_test, "Sigma Test")

## 6. Show Class Distributions

In [ ]:
datasets = {
    "Binary Train": binary_train,
    "Binary Test": binary_test,
    "Sigma Train": sigma_train,
    "Sigma Test": sigma_test,
}

class_distribution_df = build_class_distribution(datasets)
class_distribution_df

## 7. Check Sequence Length Distribution

In [ ]:
for dataset_name, df in datasets.items():
    lengths = df["sequence"].str.len()
    print(f"{dataset_name}: min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.2f}")

## 8. Check Invalid DNA Characters

In [ ]:
for dataset_name, df in datasets.items():
    report = check_dataset(df)
    print(
        f"{dataset_name}: invalid_sequence_count={report['invalid_sequence_count']}, "
        f"invalid_characters={report['invalid_characters']}"
    )

## 9. Save Dataset Summary

In [ ]:
table_dir, figure_dir = ensure_output_dirs()

summary_df = build_dataset_summary(datasets)
class_distribution_df = build_class_distribution(datasets)

summary_path = table_dir / "dataset_summary.csv"
class_distribution_path = table_dir / "dataset_class_distribution.csv"
binary_plot_path = figure_dir / "binary_class_distribution.png"
sigma_plot_path = figure_dir / "sigma_class_distribution.png"

summary_df.to_csv(summary_path, index=False)
class_distribution_df.to_csv(class_distribution_path, index=False)
plot_class_distribution(binary_train, "Binary Train Class Distribution", binary_plot_path)
plot_class_distribution(sigma_train, "Sigma Train Class Distribution", sigma_plot_path)

print("Saved:", summary_path)
print("Saved:", class_distribution_path)
print("Saved:", binary_plot_path)
print("Saved:", sigma_plot_path)

summary_df